# Synthetic Data Augmentation with Conditional Flow Matching

## Part 2

In this section, we fine-tune a pretrained ResNet classifier on the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset, using validation loss as the evaluation metric to establish a performance baseline. The model will then be evaluated with synthetic data augmentation techniques to assess their impact on classification performance. By fine-tuning the pretrained ResNet, we aim to leverage its pretrained features while adapting it to the specifics of fashion item classification.

## Setup

In [1]:
!find . -mindepth 1 -exec rm -rf {} + &> /dev/null
!git clone https://github.com/ZhangLyndon/FlowMatchingAugmentation . &> /dev/null

In [2]:
!pip install -qU -r requirements.txt

In [3]:
import os
import argparse

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torchvision
import numpy as np

# Components for initializing an ImageNet-pretrained ResNet-18 classifier, fine-
# tuning it on Fashion MNIST, and evaluating classification performance on base-
# line, low-data, and synthetically augmented settings.
from classification import (ClassificationTrainer,
                            create_classifier, ResNetClassifier,
                            SyntheticDataGenerator, SyntheticAugmentationEvaluator,
                            create_augmented_dataset)

# Utilities for loading the Fashion MNIST dataset, computing top-k categorical
# accuracy and average cross-entropy loss, and saving training results.
from utils import get_dataloaders, AverageMeter, accuracy, save_results

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["font.family"] = "DejaVu Sans Mono"

We now leverage our image generation capability for synthetic data augmentation to enhance classifier performance, particularly in low-data scenarios.

We initialize a ResNet-18 classifier pre-trained on ImageNet, replacing its final fully connected layer with a dropout-regularized linear classifier adapted to our 10-class task (the null class is excluded, as it is irrelevant for purposes of classification). In addition, the initial convolution layer is modified to process single-channel grayscale images, instead of the usual 3-channel RGB input.

To prevent overfitting, we partition the training split of the Fashion MNIST dataset, using 80% for training and 20% for validation. We monitor the cross-entropy loss on the validation set to determine the optimal number of epochs to train for before overfitting occurs.

In [4]:
# Configure classifier training pipeline
classification_args = argparse.Namespace(data_root = "./data",
                                         batch_size = 16,
                                         num_workers = 0,
                                         epochs = 30,
                                         lr = 0.001,
                                         weight_decay = 1e-4,
                                         step_size = 15,
                                         gamma = 0.1,
                                         classification_dir = "./results/classification",
                                         checkpoint_dir = "./checkpoints",
                                         save_interval = 10,
                                         seed = 42)

# Set random seed for reproducibility
torch.manual_seed(classification_args.seed)
np.random.seed(classification_args.seed)

# Create directories for classification results and model checkpoints
os.makedirs(classification_args.classification_dir, exist_ok = True)
os.makedirs(classification_args.checkpoint_dir, exist_ok = True)

In [5]:
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

train_transform = transforms.Compose([transforms.Resize((32, 32)),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean = [0.28604],
                                                           std = [0.35302])])

train_dataset = datasets.FashionMNIST(root = classification_args.data_root,
                                      train = True,
                                      download = True,
                                      transform = train_transform)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

# Randomly split the training set into non-overlapping training and validation
# subsets
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create DataLoaders from the training and validation subsets
train_loader = DataLoader(train_subset,
                          batch_size = classification_args.batch_size,
                          shuffle = True,
                          num_workers = classification_args.num_workers)

val_loader = DataLoader(val_subset,
                        batch_size = classification_args.batch_size,
                        shuffle = False,
                        num_workers = classification_args.num_workers)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 206kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.82MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.6MB/s]


In [6]:
# Initialize the ResNet-18 trainer on Fashion MNIST. Monitor performance using
# the validation set.
trainer = ClassificationTrainer(classification_args)
results = trainer.train(train_loader, val_loader)

Number of epochs: 30
Number of training samples: 48000
Number of validation samples: 12000


Validation: 100%|██████████| 750/750 [00:07<00:00, 100.13it/s]


Training Set | Loss: 0.7028, Top-1 Accuracy: 76.42%, Top-5 Accuracy: 98.74%
Validation Set | Loss: 0.4749, Top-1 Accuracy: 81.47%, Top-5 Accuracy: 99.57%
Best Validation Loss (Up Until Now): 0.4749
_________________________________________________________________________________________________________



Validation: 100%|██████████| 750/750 [00:07<00:00, 102.08it/s]


Training Set | Loss: 0.4621, Top-1 Accuracy: 84.08%, Top-5 Accuracy: 99.55%
Validation Set | Loss: 0.3816, Top-1 Accuracy: 85.69%, Top-5 Accuracy: 99.67%
Best Validation Loss (Up Until Now): 0.3816
_________________________________________________________________________________________________________



...

Training Set | Loss: 0.0935, Top-1 Accuracy: 96.63%, Top-5 Accuracy: 100.00%
Validation Set | Loss: 0.2396, Top-1 Accuracy: 92.58%, Top-5 Accuracy: 99.90%
Best Validation Loss (Up Until Now): 0.2141
_________________________________________________________________________________________________________



Validation: 100%|██████████| 750/750 [00:07<00:00, 106.16it/s]


Training Set | Loss: 0.0893, Top-1 Accuracy: 96.89%, Top-5 Accuracy: 100.00%
Validation Set | Loss: 0.2468, Top-1 Accuracy: 92.07%, Top-5 Accuracy: 99.94%
Best Validation Loss (Up Until Now): 0.2141
_________________________________________________________________________________________________________



Validation: 100%|██████████| 750/750 [00:06<00:00, 114.64it/s]


Training Set | Loss: 0.0849, Top-1 Accuracy: 96.95%, Top-5 Accuracy: 100.00%
Validation Set | Loss: 0.2489, Top-1 Accuracy: 92.33%, Top-5 Accuracy: 99.89%
Best Validation Loss (Up Until Now): 0.2141
_________________________________________________________________________________________________________

Training Time: 3552.87 seconds
Best Validation Loss (All Epochs): 0.2141


Epoch 21 showed the lowest validation loss (0.2141), with nearby epochs (16-25) clustering tightly between 0.2141 and 0.2267. To ensure consistency and fairness in cross-dataset comparison, we will standardize subsequent runs to 25 epochs.